# Day 29: LLM Fundamentals & Prompt Engineering

**Week 5 — Agent Development**

---

## 📚 Theory: Understanding the Brain of the Agent

Before building multi-agent orchestrators, you must deeply understand the underlying primitive: the Large Language Model (LLM).

### Tokenization & Context Windows
- LLMs do not read words; they read **tokens** (roughly 0.75 of a word). `"Hamburger"` might be split into `"Ham"`, `"bur"`, `"ger"`.
- The **Context Window** is the maximum number of tokens an LLM can process in a single request (both input + output). E.g., Gemini 1.5 Pro has a massive 2 Million token context window.
- Exceeding the context window results in an error. Managing this constraint is a core part of Agent Development.

### System Prompts vs User Prompts
- **System Prompt (Developer Instructions)**: Defines the persona, constraints, and available tools. *(e.g., "You are a helpful coding assistant. Do not use external libraries.")*
- **User Prompt**: The actual request from the human user.

### Advanced Prompting Techniques
1. **Zero-Shot Prompting**: Asking the model to perform a task without giving examples.
2. **Few-Shot Prompting**: Providing 2-3 examples of Input->Output pairs in the prompt to "steer" the model's format.
3. **Chain-of-Thought (CoT)**: Forcing the model to "think aloud" before outputting the final answer (e.g., adding *"Let's think step by step."*). This significantly improves reasoning accuracy because it allows the model to "generate" more tokens (computation) before committing to a final answer.

## 🔨 Exercise 1: Building a Few-Shot Prompt Template

Let's simulate how an SDK constructs prompts before sending them to an LLM API.

**Task**: Write a Python function `build_sentiment_prompt(user_text)` that constructs a Few-Shot prompt for classifying text as `POSITIVE`, `NEGATIVE`, or `NEUTRAL`.

In [ ]:
def build_sentiment_prompt(user_text: str) -> str:
    """
    Constructs a Few-Shot prompt string.
    """
    # YOUR CODE HERE
    pass

# Test your prompt structure
prompt = build_sentiment_prompt("The battery life on this phone is terrible.")
print(prompt)

In [ ]:
# Solution
def build_sentiment_prompt_solution(user_text: str) -> str:
    system_instruction = "Classify the sentiment of the following text as POSITIVE, NEGATIVE, or NEUTRAL. Return ONLY the classification word."
    
    few_shot_examples = """
Text: "I absolutely love the new design!"
Sentiment: POSITIVE

Text: "The food was bland and overpriced."
Sentiment: NEGATIVE

Text: "The package arrived on Tuesday."
Sentiment: NEUTRAL
"""
    
    final_prompt = f"{system_instruction}\n{few_shot_examples}\nText: \"{user_text}\"\nSentiment: "
    return final_prompt

print("Solution Output:")
print(build_sentiment_prompt_solution("The battery life on this phone is terrible."))


## 🔨 Exercise 2: Implementing a Context Window Truncator

In real Agent systems, memory (chat history) grows infinitely. If you exceed the token limit, the API call fails. 

**Task**: Write a function `trim_chat_history(history, max_tokens)` that takes a list of message dictionaries and removes the *oldest* messages (except the very first System Prompt) until the total estimated tokens are under `max_tokens`.
*(Assume 1 word = 1.3 tokens for this mock exercise).* 

In [ ]:
def estimate_tokens(text: str) -> int:
    return int(len(text.split()) * 1.3)

def trim_chat_history(history: list[dict], max_tokens: int) -> list[dict]:
    """
    history format: [{'role': 'system', 'content': '...'}, {'role': 'user', 'content': '...'}]
    Ensure the first message (system prompt) is NEVER deleted.
    """
    # YOUR CODE HERE
    pass

mock_history = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris. It is known for the Eiffel Tower and the Louvre Museum."},
    {"role": "user", "content": "Tell me more about the Louvre."}
]

# Test: Trim to a very small token limit
# res = trim_chat_history(mock_history, 15)
# print(res)

In [ ]:
# Solution
def trim_chat_history_solution(history: list[dict], max_tokens: int) -> list[dict]:
    if not history:
        return []
        
    # Keep the system prompt safe
    system_prompt = history[0]
    chat_history = history[1:]
    
    def get_total_tokens(sys, chat):
        total = estimate_tokens(sys["content"])
        for msg in chat:
            total += estimate_tokens(msg["content"])
        return total
        
    # Pop from the beginning (oldest) of the chat history until we fit
    while chat_history and get_total_tokens(system_prompt, chat_history) > max_tokens:
        chat_history.pop(0)  # Remove oldest message
        
    return [system_prompt] + chat_history

print("Solution Output:")
print(trim_chat_history_solution(mock_history, 25))


## 📝 Day 29 Review

### Concepts Validated Today
- [ ] How Tokens relate to Context Windows.
- [ ] Formatting prompts using **Few-Shot** and **Chain-of-Thought** techniques.
- [ ] Programmatically managing Agent memory by truncating old chat history to avoid Token Limit errors.

### Tomorrow's Preview
**Day 30: Agent Architectures** — We upgrade from simple LLM calls to "Agents". We will learn the ReAct framework, state machines, and routing between multiple specialized agents.